In [1]:
import numpy as numpy
import pandas as pd
import difflib
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings('ignore')

In [ ]:
PATH = "./data.csv"
data = pd.read_csv(PATH)
data.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6810 entries, 0 to 6809
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   isbn13          6810 non-null   int64  
 1   isbn10          6810 non-null   object 
 2   title           6810 non-null   object 
 3   subtitle        2381 non-null   object 
 4   authors         6738 non-null   object 
 5   categories      6711 non-null   object 
 6   thumbnail       6481 non-null   object 
 7   description     6548 non-null   object 
 8   published_year  6804 non-null   float64
 9   average_rating  6767 non-null   float64
 10  num_pages       6767 non-null   float64
 11  ratings_count   6767 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 638.6+ KB


In [4]:
data['isbn13'] = str(data['isbn13'])
data['categories'] = data['categories'].fillna('None')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6810 entries, 0 to 6809
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   isbn13          6810 non-null   object 
 1   isbn10          6810 non-null   object 
 2   title           6810 non-null   object 
 3   subtitle        2381 non-null   object 
 4   authors         6738 non-null   object 
 5   categories      6810 non-null   object 
 6   thumbnail       6481 non-null   object 
 7   description     6548 non-null   object 
 8   published_year  6804 non-null   float64
 9   average_rating  6767 non-null   float64
 10  num_pages       6767 non-null   float64
 11  ratings_count   6767 non-null   float64
dtypes: float64(4), object(8)
memory usage: 638.6+ KB


In [6]:
relevant_titles = ['title', 'authors', 'categories', 'average_rating', 'num_pages', 'ratings_count']
relevant_titles_str = [x for x in relevant_titles if data[x].dtype == 'object']
relevant_titles_num = [x for x in relevant_titles if not x in relevant_titles_str]
selected_features = data[relevant_titles]

In [7]:
relevant_titles_num

['average_rating', 'num_pages', 'ratings_count']

In [8]:
data.describe()

,published_year,average_rating,num_pages,ratings_count
count,6804.000000,6767.000000,6767.000000,6.767000e+03
mean,1998.630364,3.933284,348.181026,2.106910e+04
std,10.484257,0.331352,242.376783,1.376207e+05
min,1853.000000,0.000000,0.000000,0.000000e+00
25%,1996.000000,3.770000,208.000000,1.590000e+02
50%,2002.000000,3.960000,304.000000,1.018000e+03
75%,2005.000000,4.130000,420.000000,5.992500e+03
max,2019.000000,5.000000,3342.000000,5.629932e+06


#### Genres

### Data Processing

In [9]:
for feature in relevant_titles_str:
    selected_features[feature] = selected_features[feature].fillna('')
for feature in relevant_titles_num:
    selected_features[feature] = selected_features[feature].fillna(0)

In [10]:
combined_features = selected_features['authors'] + ' ' + selected_features['title'] + ' ' + selected_features['categories']

In [11]:
selected_features.isna().sum()

title             0
authors           0
categories        0
average_rating    0
num_pages         0
ratings_count     0
dtype: int64

In [12]:
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)

In [13]:
similarity = cosine_similarity(feature_vectors, feature_vectors)

In [14]:
list_of_all_titles = selected_features['title'].to_list()
list_of_all_genres = data['categories'].to_list()

In [15]:
book_name = input('Enter your favourite Book name: ')
type(book_name)

Enter your favourite Book name:  Wuthering heights


str

In [16]:
find_close_match = difflib.get_close_matches(book_name, list_of_all_titles)
print(find_close_match)

['Wuthering Heights', 'Wuthering Heights', 'Wuthering Heights']


In [17]:
close_match = find_close_match[0]
index_of_the_book = selected_features[selected_features.title == close_match].index[0]

In [18]:
similarity_score = list(enumerate(similarity[index_of_the_book]))

In [19]:
sorted_similar_books = sorted(similarity_score, key = lambda x: x[1], reverse=True)
print(sorted_similar_books)

[(856, 1.0000000000000002), (2453, 0.8703852349478035), (3064, 0.8703852349478035), (1005, 0.27690004878500996), (3420, 0.2627922490745036), (1519, 0.23050961593846234), (2731, 0.22550646139083383), (6260, 0.22536573334070406), (1370, 0.22490435850808355), (1516, 0.22117837022220282), (1517, 0.22117837022220282), (1525, 0.21877667288328487), (5664, 0.20337778271566448), (2175, 0.2010088514254994), (495, 0.1846267592758187), (4595, 0.16992841393705044), (3903, 0.14352788950737885), (5297, 0.14182150382134168), (5766, 0.03306452696545746), (1196, 0.026192571560851075), (3005, 0.02322211305670495), (2328, 0.02259313496664116), (1456, 0.022499309707019435), (1422, 0.02013337200210638), (350, 0.019906186594019354), (6645, 0.019764306065724345), (691, 0.019719326053879525), (3417, 0.019487057916950247), (2979, 0.019174602165272007), (2059, 0.018832791068868163), (5832, 0.018803913588886716), (3253, 0.0185152634500158), (4104, 0.018402937537964237), (3837, 0.01812126450772219), (1130, 0.01811

In [20]:
top_sim = sorted_similar_books[:5]
top_sim

[(856, 1.0000000000000002),
 (2453, 0.8703852349478035),
 (3064, 0.8703852349478035),
 (1005, 0.27690004878500996),
 (3420, 0.2627922490745036)]

In [21]:
books = []
for book in top_sim:
    index = book[0]
    book = selected_features[selected_features.index == index]['title'].values[0]
    books.append(book)
books

['Wuthering Heights',
 'Wuthering Heights',
 'Wuthering Heights',
 'Jane Eyre',
 'Emma']

In [38]:
idm = top_sim[0][0]
selected_features[selected_features.index == idm]['title']

RangeIndex(start=0, stop=6810, step=1)

In [49]:
book_genre_ds = data[data['title'] == books[0]]
book_genre_ds_index = book_genre_ds.index
book_index = book_genre_ds_index[0]
book_genre = book_genre_ds['categories'][book_index]
book_author = book_genre_ds['authors'][book_index]
book_author

'Emily Brontë'

In [50]:
book_genre

'Fiction'

In [54]:
close_genre_match = difflib.get_close_matches(book_genre, list_of_all_genres)

In [55]:
close_genre_match

['Fiction', 'Fiction', 'Fiction']

In [56]:
similar_genres = data[data['categories'] == close_genre_match[0]]
similar_authors = data

In [57]:
similar_genres['average_rating']

0       3.85
3       3.93
12      4.03
54      3.71
60      4.14
        ... 
6773    3.79
6776    3.92
6778    3.77
6797    3.75
6799    3.22
Name: average_rating, Length: 2588, dtype: float64

In [58]:
def filter_authors(df):
    # Ensure 'book_author' and 'book_genre' are defined before calling this function
    target_authors = df[df['authors'] == book_author]
    target_categories = df[df['categories'] == book_genre]

    # Assign scores based on matches
    df['author_score'] = df['authors'].apply(lambda x: 0.5 if x in target_authors['authors'].values else 0)
    df['genre_score'] = df['categories'].apply(lambda x: 0.5 if x in target_categories['categories'].values else 0)

    return df

In [59]:
def filter_authors(df):
    target_authors = df[df['authors'] == book_author]
    target_categories = df[df['categories'] == book_genre]

    df['author_score'] = df['authors'].apply(lambda x: 0.5 if x in target_authors['authors'].values else 0)
    df['genre_score'] = df['categories'].apply(lambda x: 0.5 if x in target_categories['categories'].values else 0)

    return df


In [60]:
data = filter_authors(data)

In [61]:
data['average_rating'] = data['average_rating'].fillna(0)
data['ratings_count'] = data['num_pages'].fillna(0)
data['num_pages'] = data['ratings_count'].fillna(0)

In [62]:
average_rating_highest = data['average_rating'].max()
ratings_count_highest = data['ratings_count'].max()
data['average_rating_scaled'] = (data['average_rating']/average_rating_highest)*1.5
data['ratings_count_scaled'] = (data['ratings_count']/ratings_count_highest)*1.0
data['ratings'] = data['author_score'] + data['average_rating_scaled'] + data['ratings_count_scaled'] + data['genre_score']

In [63]:
data.isna().sum()

isbn13                      0
isbn10                      0
title                       0
subtitle                 4429
authors                    72
categories                  0
thumbnail                 329
description               262
published_year              6
average_rating              0
num_pages                   0
ratings_count               0
author_score                0
genre_score                 0
average_rating_scaled       0
ratings_count_scaled        0
ratings                     0
dtype: int64

In [64]:
data.nunique()
data['author_score'].unique()

array([0. , 0.5])

In [65]:
sorted_data = data.sort_values(by='ratings', ascending=False)
sorted_data['ratings']

5197    2.529071
2730    2.419000
1937    2.394056
5438    2.351970
3721    2.316693
          ...   
459     0.000000
460     0.000000
884     0.000000
4375    0.000000
528     0.000000
Name: ratings, Length: 6810, dtype: float64

In [66]:
recommended_books = sorted_data['title'].head(10)
recommended_books = list(recommended_books)

In [67]:
recommended_books

['The Sword of Truth',
 'The Harry Potter Collection',
 'The Hobbit / The Lord of the Rings',
 'Complete Works',
 'The History of the Lord of the Rings',
 'The Norton Anthology of Short Fiction',
 'The Belgariad Boxed Set',
 'The Complete Novels',
 'The Civil War, a Narrative',
 'Harry Potter']